# S2d mechanism-shortcut pilot (CPU only)

Before **Save Version -> Save & Run All**, attach only the existing Kaggle dataset `mintesnotfikir/cdd-11-30` and enable Internet for the pinned Git clone. Use the default CPU session; no GPU and no pretrained-model input are required. This notebook never downloads image data, never enumerates or reads `CDD-11_test`, and never trains or loads a restoration model. It regenerates the locked 25-scene A/B cache, exports raw A/B/GT files with hashes, and runs scene-grouped mechanism probes.


In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import shutil
import subprocess
import sys
import traceback
from pathlib import Path

PROJECT_URL = 'https://github.com/HoangKhanhTung0111/CoT-restoration.git'
PROJECT_COMMIT = 'ae8ee0a3f326500bb673223e30df688fea6ce4da'
CDD11_ROOT = Path('/kaggle/input/datasets/mintesnotfikir/cdd-11-30')
WORK = Path('/kaggle/working/s2d_mechanism_probe')
PROJECT = WORK / 'project'
PREPARED = WORK / 'prepared'
BUNDLE = WORK / 'bundle'
PROBE = BUNDLE / 'probe'
LOGS = BUNDLE / 'logs'
MANIFEST = PREPARED / 'cv_manifest.json'
CACHE = PREPARED / 'generated_cache'
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ARCHIVE = Path('/kaggle/working') / f's2d_mechanism_probe_{RUN_ID}.zip'
errors = []
stage = 'initialized'
BUNDLE.mkdir(parents=True, exist_ok=False)
LOGS.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def run_logged(command, name, cwd=None):
    command = [str(item) for item in command]
    print('+', ' '.join(command), flush=True)
    with (LOGS / f'{name}.log').open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f'{name} failed with exit code {code}')

def record_failure():
    errors.append(f'Stage {stage} failed:\n' + traceback.format_exc())
    print(errors[-1], flush=True)


In [ ]:
try:
    stage = 'attached_input_preflight'
    assert CDD11_ROOT.is_dir(), f'Missing attached input: {CDD11_ROOT}'
    print('Attached data:', CDD11_ROOT)

    stage = 'pinned_code_setup'
    run_logged(['git', 'clone', PROJECT_URL, PROJECT], 'clone_project')
    run_logged(['git', '-C', PROJECT, 'checkout', '--detach', PROJECT_COMMIT], 'checkout_project')
    actual = subprocess.check_output(['git', '-C', PROJECT, 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == PROJECT_COMMIT
    run_logged([sys.executable, '-m', 'unittest', 'hybrid_cot_nafnet.test_low_weather_generator_b', 'hybrid_cot_nafnet.test_s2b_coverage', 'hybrid_cot_nafnet.test_s2d_mechanism_probe', '-v'], 'unit_tests', cwd=PROJECT)
except Exception:
    record_failure()


In [ ]:
if not errors:
    try:
        stage = 'prepare_locked_manifest_and_ab_cache'
        run_logged([sys.executable, '-u', '-m', 'hybrid_cot_nafnet.prepare_s2b_coverage', '--data-root', CDD11_ROOT, '--manifest', MANIFEST, '--cache-root', CACHE], 'prepare', cwd=PROJECT)
    except Exception:
        record_failure()


In [ ]:
if not errors:
    try:
        stage = 'scene_grouped_mechanism_probe'
        run_logged([sys.executable, '-u', '-m', 'hybrid_cot_nafnet.s2d_mechanism_probe', '--data-root', CDD11_ROOT, '--manifest', MANIFEST, '--cache-root', CACHE, '--output-dir', PROBE, '--export-raw'], 'probe', cwd=PROJECT)
    except Exception:
        record_failure()


In [ ]:
try:
    stage_before_export = stage
    for source in (MANIFEST, CACHE / 'cache_manifest.json'):
        if source.is_file():
            shutil.copy2(source, BUNDLE / source.name)
    run_data = {
        'run_id_utc': RUN_ID,
        'status': 'COMPLETE' if not errors else 'FAILED_OR_PARTIAL',
        'stage': stage_before_export,
        'errors': errors,
        'project_commit': PROJECT_COMMIT,
        'data_input': str(CDD11_ROOT),
        'pretrained_input': None,
        'dataset_or_checkpoint_downloaded': False,
        'cdd11_test_opened': False,
        'restoration_checkpoint_loaded': False,
        'restoration_training_performed': False,
    }
    (BUNDLE / 'run.json').write_text(json.dumps(run_data, indent=2) + '\n', encoding='utf-8')
    if ARCHIVE.exists():
        ARCHIVE.unlink()
    shutil.make_archive(str(ARCHIVE.with_suffix('')), 'zip', root_dir=BUNDLE)
    print('Download artifact:', ARCHIVE)
    print('SHA256:', sha256_file(ARCHIVE))
    print('Status:', run_data['status'])
except Exception:
    record_failure()
    raise
if errors:
    raise RuntimeError(f'S2d pilot failed; download {ARCHIVE} for audit')
